# Chow Test for a Known Structural Break

For the Chow test, we can implement the classical **F-test** by comparing:

1. A **pooled regression** using all observations.
2. **Separate regressions** before and after the known break point.

The test then determines whether the regression coefficients changed significantly.


## 1. Python implementation


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import f


def chow_test(
    y,
    X,
    break_point,
    significance=0.05,
    name="Regression"
):
    """
    Perform the Chow test for a known structural break.

    Parameters
    ----------
    y : array-like
        Dependent variable.

    X : DataFrame or array-like
        Independent variables.

    break_point : int
        Observation index where the structural break occurs.
        Observations before break_point belong to period 1,
        observations from break_point onward belong to period 2.

    significance : float
        Significance level, e.g. 0.05.

    name : str
        Name of the regression.

    Returns
    -------
    dict
        Chow statistic, p-value, decision and conclusion.
    """

    # Convert data
    y = np.asarray(y, dtype=float)

    if isinstance(X, pd.DataFrame):
        X = X.copy()
    else:
        X = pd.DataFrame(X)

    # Add intercept
    X = sm.add_constant(X, has_constant="add")

    # Remove missing observations
    data = pd.concat(
        [
            pd.Series(y, name="y"),
            X.reset_index(drop=True)
        ],
        axis=1
    ).dropna()

    y = data["y"].values
    X = data.drop(columns="y").values

    n = len(y)
    k = X.shape[1]  # includes intercept

    # Validate break point
    if break_point <= k:
        raise ValueError(
            "Break point is too early for the regression."
        )

    if n - break_point <= k:
        raise ValueError(
            "Not enough observations after the break point."
        )

    # --------------------------------------------------
    # 1. Pooled regression
    # --------------------------------------------------

    pooled_model = sm.OLS(
        y,
        X
    ).fit()

    RSS_pooled = np.sum(
        pooled_model.resid ** 2
    )

    # --------------------------------------------------
    # 2. Regression before break
    # --------------------------------------------------

    y_before = y[:break_point]
    X_before = X[:break_point]

    model_before = sm.OLS(
        y_before,
        X_before
    ).fit()

    RSS_before = np.sum(
        model_before.resid ** 2
    )

    # --------------------------------------------------
    # 3. Regression after break
    # --------------------------------------------------

    y_after = y[break_point:]
    X_after = X[break_point:]

    model_after = sm.OLS(
        y_after,
        X_after
    ).fit()

    RSS_after = np.sum(
        model_after.resid ** 2
    )

    # --------------------------------------------------
    # 4. Chow F-statistic
    # --------------------------------------------------

    numerator = (
        RSS_pooled -
        (RSS_before + RSS_after)
    ) / k

    denominator = (
        (RSS_before + RSS_after)
        / (n - 2 * k)
    )

    chow_statistic = numerator / denominator

    # p-value
    p_value = f.sf(
        chow_statistic,
        k,
        n - 2 * k
    )

    # --------------------------------------------------
    # 5. Decision
    # --------------------------------------------------

    if p_value <= significance:

        decision = "Reject H0"

        conclusion = (
            "There is statistically significant evidence "
            "of a structural break at the specified "
            "break point."
        )

    else:

        decision = "Fail to Reject H0"

        conclusion = (
            "There is insufficient evidence of a "
            "structural break at the specified "
            "break point."
        )

    # --------------------------------------------------
    # 6. Display results
    # --------------------------------------------------

    print("=" * 70)
    print("CHOW TEST")
    print("=" * 70)

    print(f"Regression          : {name}")
    print(f"Observations        : {n}")
    print(f"Break point         : {break_point}")
    print(f"Significance level  : {significance}")

    print("\nHypotheses:")
    print("H0: No structural break.")
    print("    Regression coefficients are stable.")
    print("H1: Structural break exists.")
    print("    Regression coefficients changed.")

    print("\nResidual Sum of Squares:")
    print(f"Pooled regression   : {RSS_pooled:.6f}")
    print(f"Before break        : {RSS_before:.6f}")
    print(f"After break         : {RSS_after:.6f}")

    print("\nChow Test Result:")
    print(f"F-statistic         : {chow_statistic:.4f}")
    print(f"p-value             : {p_value:.6f}")

    print("\nDecision:")
    print(decision)

    print("\nConclusion:")
    print(conclusion)

    print("=" * 70)

    return {
        "chow_statistic": chow_statistic,
        "p_value": p_value,
        "decision": decision,
        "conclusion": conclusion,
        "rss_pooled": RSS_pooled,
        "rss_before": RSS_before,
        "rss_after": RSS_after
    }

## 2. Example: create data with a structural break

Let's create a regression where the relationship between X and Y changes after observation 100.

**Before the break:**  
$Y = 10 + 2X + \epsilon$

**After the break:**  
$Y = 20 + 0.5X + \epsilon$


In [ ]:
np.random.seed(42)

n = 200

X = np.random.normal(size=n)

noise = np.random.normal(scale=1, size=n)

Y = np.zeros(n)

# Before break
Y[:100] = (
    10
    + 2 * X[:100]
    + noise[:100]
)

# After break
Y[100:] = (
    20
    + 0.5 * X[100:]
    + noise[100:]
)

print(f"Total observations: {n}")
print(f"Break point: 100")
print(f"True relationship changes at observation 100")

Now run the Chow test:


In [ ]:
result = chow_test(
    y=Y,
    X=pd.DataFrame({
        "X": X
    }),
    break_point=100,
    significance=0.05,
    name="Y ~ X"
)

Because we deliberately changed both the intercept and slope at observation 100, you should get a very small p-value (Reject H₀).


### Optional: Visualize the data and fitted lines


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

# Scatter
ax.scatter(X[:100], Y[:100], alpha=0.6, label="Before break (obs 0–99)", color="steelblue")
ax.scatter(X[100:], Y[100:], alpha=0.6, label="After break (obs 100–199)", color="crimson")

# True lines (approximate)
x_line = np.linspace(X.min(), X.max(), 100)
ax.plot(x_line, 10 + 2 * x_line, color="navy", linestyle="--", linewidth=2, label="True: Y=10+2X")
ax.plot(x_line, 20 + 0.5 * x_line, color="darkred", linestyle="--", linewidth=2, label="True: Y=20+0.5X")

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Simulated Data with Structural Break at Observation 100")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Example without a structural break

Now create a **stable** relationship (no break):


In [ ]:
np.random.seed(42)

n = 200

X_stable = np.random.normal(size=n)

noise_stable = np.random.normal(scale=1, size=n)

Y_stable = (
    10
    + 2 * X_stable
    + noise_stable
)

print(f"Total observations: {n}")
print("True relationship is stable across the entire sample")

In [ ]:
result_stable = chow_test(
    y=Y_stable,
    X=pd.DataFrame({
        "X": X_stable
    }),
    break_point=100,
    significance=0.05,
    name="Stable Regression"
)

You will generally obtain a relatively large p-value and **Fail to Reject H₀**.


## 4. Decision rule

The Chow test uses:

| Hypothesis | Statement |
|------------|-----------|
| **H₀** | No structural break (coefficients are stable) |
| **H₁** | Structural break exists (coefficients changed) |

At significance level **α = 0.05**:

| p-value | Decision | Interpretation |
|---------|----------|----------------|
| p ≤ 0.05 | Reject H₀ | Significant structural break |
| p > 0.05 | Fail to reject H₀ | No significant break detected |


## 5. Financial example

Suppose you want to test whether the relationship between a stock and NIFTY changed after a particular event.

Your regression is:

$$R_{\text{stock},t} = \alpha + \beta R_{\text{NIFTY},t} + \epsilon_t$$

Your data might look like:

```python
df = pd.read_csv("stock_data.csv")

df["Stock_Return"] = df["Stock_Close"].pct_change()
df["Nifty_Return"] = df["Nifty_Close"].pct_change()

df = df.dropna()
```

Suppose observation **500** corresponds to your suspected event:

```python
result = chow_test(
    y=df["Stock_Return"],
    X=df[["Nifty_Return"]],
    break_point=500,
    significance=0.05,
    name="Stock Return ~ NIFTY Return"
)
```

If the result is something like:

```
F-statistic : 4.72
p-value     : 0.009
```

then:

$$0.009 < 0.05$$

Therefore: **Reject H₀**.

**Conclusion:** There is significant evidence that the stock's relationship with NIFTY changed at the specified break point. In other words, the estimated alpha and/or beta changed.


### Demo: Simulated stock–market relationship with a break


In [ ]:
np.random.seed(7)

n = 400
market = np.random.normal(0.0005, 0.012, n)

# Before: beta ≈ 1.2, After: beta ≈ 0.6
stock = np.zeros(n)
stock[:200] = 0.0002 + 1.2 * market[:200] + np.random.normal(0, 0.008, 200)
stock[200:] = 0.0005 + 0.6 * market[200:] + np.random.normal(0, 0.008, 200)

result_fin = chow_test(
    y=stock,
    X=pd.DataFrame({"Market": market}),
    break_point=200,
    significance=0.05,
    name="Stock ~ Market (simulated)"
)

---
**Note:** The classical Chow test assumes homoskedastic, independent errors. In financial time series, heteroskedasticity and autocorrelation are common, so robust versions or HAC-standard-error approaches may be preferable for formal inference.
